# Lip 실행 예제
REINVENT4 기반 생성 설계 파이프라인.
## 실습 개요
- **활용 분야** — 분자 생성
- **카테고리** — Screening · Property
- **모델 유형** — REINVENT (RNN/Transformer)
- **발표 연도** — 2024
- **라이선스** — Apache-2.0
**관련 논문**

- [REINVENT 4: Modern AI-driven generative molecule design](https://jcheminf.biomedcentral.com/articles/10.1186/s13321-024-00812-5) — Hannes H. Löffler 외, Journal of Cheminformatics, 2024

## 실습 목표

공결정 리간드가 포함된 사람 ABL 구조로 Lip의 짧은 분자 생성·평가 실습을 실행합니다. 의약품 효능 검증은 아닙니다.

## 실행 조건

Colab의 x86_64 Linux 런타임에서 공개 모델 이미지를 udocker로 실행합니다. Docker 데몬과 플랫폼 로그인은 필요하지 않습니다.

**GPU 런타임이 필요합니다.** GPU를 배정받지 못하면 이미지 다운로드 전에 중단합니다. 입력과 모델에 따라 메모리 부족이 발생할 수 있습니다.

압축 이미지 약 5.9 GiB. 다운로드·압축 해제를 위해 약 19.6 GiB의 여유 공간을 사전 확인합니다. 모델별로 한 노트북씩 실행하세요.

Image: `ghcr.io/connects-scv/laidd-lip@sha256:018c3836fa5827725798ffea5e97416fed306241149ba5411d52cc19514e24fa`

플랫폼 카탈로그 버전: 미기재. 이 노트북은 위 이미지 digest의 모델·가중치를 사용합니다. 플랫폼과 다른 버전이면 결과가 같다고 가정하지 마세요.

Runtime documentation: https://indigo-dc.github.io/udocker/user_manual.html

세션 종료 시 설치·컨테이너·결과가 사라질 수 있으므로 결과 ZIP을 내려받으세요.

## 실행 절차

런타임 → 환경 준비 → 입력값 설정 → 추론 → 결과 확인 순서로 실행합니다. 환경 준비는 첫 실행에 큰 파일을 내려받으므로 시간이 걸립니다.

## 실행 도구

In [ ]:
"""Shared runtime embedded in LAIDD Hub container-example notebooks.

Only setup() downloads software. Import, input preparation, and result validation
can be tested independently. Images are public and pinned by OCI manifest digest.
"""
import base64
import csv
import gzip
import hashlib
import io
import json
import os
import platform
import re
import shutil
import signal
import subprocess
import sys
import time
import uuid
import zipfile
from datetime import datetime, timezone
from pathlib import Path
from sysconfig import get_path


class ColabModelExample:
    def __init__(self, spec, assets, work_root=None, execution_mode="F3"):
        if execution_mode not in {"P1", "P2", "F3"}:
            raise ValueError("execution_mode must be P1, P2 or F3")
        self.spec = spec
        self.assets = assets
        self.mode = execution_mode
        root = Path(work_root or (Path('/content') if Path('/content').is_dir() else Path.cwd()))
        self.root = root / ("laidd_" + spec["slug"])
        self.root.mkdir(parents=True, exist_ok=True)
        self.run_dir = self.root / (datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '_' + uuid.uuid4().hex[:8])
        self.input_dir = self.run_dir / 'input'
        self.output_dir = self.run_dir / 'output'
        self.input_dir.mkdir(parents=True)
        self.output_dir.mkdir()
        self.env = dict(os.environ)
        self.env['UDOCKER_DIR'] = str(self.root / 'udocker')
        self.udocker = [str(Path(get_path('scripts')) / 'udocker')]
        if hasattr(os, 'geteuid') and os.geteuid() == 0:
            self.udocker.append('--allow-root')
        # udocker 1.3.17 accepts a registry digest in its tag field (repo:sha256:...).
        # Docker's repo@sha256 form is not accepted for fully-qualified repos by this release.
        self.image = spec['image'] + ':' + spec['digest']
        self.container = 'laidd-' + spec['slug'] + '-' + spec['digest'].split(':')[1][:12]
        self.parameters = None
        self.ready = False
        self.completed = False
        self.gpu_info = None
        self.image_metadata = None

    def _call(self, args, timeout=1800, capture=False):
        return subprocess.run(args, check=True, env=self.env, timeout=timeout,
                              text=True, stdout=subprocess.PIPE if capture else None,
                              stderr=subprocess.PIPE if capture else None)

    def _container_exists(self):
        if not Path(self.udocker[0]).is_file():
            return False
        return subprocess.run(self.udocker + ['inspect', self.container], env=self.env,
                              stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, timeout=30).returncode == 0

    def preflight(self, cached=False):
        if platform.system() != 'Linux' or platform.machine().lower() not in {'x86_64', 'amd64'}:
            raise RuntimeError('Use an x86_64 Linux runtime, such as Google Colab. macOS/ARM is not supported by these images.')
        required = (0 if cached else self.spec['compressed_bytes'] * 3) + 2 * 1024**3
        free = shutil.disk_usage(self.root).free
        print(f"Disk budget: {required / 1024**3:.1f} GiB; available: {free / 1024**3:.1f} GiB")
        if free < required:
            raise RuntimeError('Not enough free disk for compressed layers and image extraction. Use a runtime with more disk space.')
        if self.spec['gpu']:
            if not shutil.which('nvidia-smi'):
                raise RuntimeError('A GPU is required. Select a GPU runtime before downloading the image.')
            result = self._call(['nvidia-smi', '--query-gpu=name,memory.total,compute_cap', '--format=csv,noheader,nounits'], timeout=30, capture=True)
            self.gpu_info = result.stdout.strip()
            print('GPU:', self.gpu_info)
            device = next(csv.reader(io.StringIO(self.gpu_info)))
            capability = float(device[2])
            if capability < self.spec.get('min_compute_capability', 0):
                raise RuntimeError('This image requires an Ampere-or-newer GPU (compute capability >= 8.0), such as L4/A100. T4 is not supported.')
        return {'required_disk_bytes': required, 'free_disk_bytes': free, 'gpu': self.gpu_info}

    def setup(self):
        self.preflight(cached=self._container_exists())
        self._call([sys.executable, '-m', 'pip', 'install', 'udocker==1.3.17'])
        self._call(self.udocker + ['install'])
        exists = self._container_exists()
        if not exists:
            print('Downloading pinned image:', self.spec['image'] + '@' + self.spec['digest'], flush=True)
            self._call(self.udocker + ['pull', '--platform=linux/amd64', self.image], timeout=5400)
            self._call(self.udocker + ['create', '--name=' + self.container, self.image], timeout=5400)
        self._call(self.udocker + ['setup', '--execmode=' + self.mode, self.container], timeout=600)
        if self.spec['gpu']:
            self._call(self.udocker + ['setup', '--nvidia', self.container], timeout=600)
            if self.mode.startswith('F'):
                self._call(self.udocker + ['setup', '--execmode=' + self.mode, '--force', self.container], timeout=600)
            probe = "import ctypes; c=ctypes.CDLL('libcuda.so.1'); assert c.cuInit(0)==0, 'CUDA driver init failed'; n=ctypes.c_int(); assert c.cuDeviceGetCount(ctypes.byref(n))==0 and n.value>0, 'GPU is not visible inside the container'; print('Container CUDA devices:', n.value)"
            self._call(self.command(argv=['python3', '-c', probe]), timeout=180)
        self.image_metadata = self._call(self.udocker + ['inspect', self.container], capture=True).stdout
        self.ready = True
        print('Environment ready. Input files:', self.input_dir)

    @staticmethod
    def _sequence(value):
        sequence = re.sub(r'\s+', '', str(value)).upper()
        if not sequence or re.search('[^ACDEFGHIKLMNPQRSTVWY]', sequence):
            raise ValueError('Use a non-empty amino acid sequence containing standard one-letter codes.')
        return sequence

    def prepare_inputs(self, parameters):
        p = dict(parameters)
        for filename, key in self.spec['assets'].items():
            asset = self.assets[key]
            raw = gzip.decompress(base64.b64decode(asset['gzip_base64']))
            if hashlib.sha256(raw).hexdigest() != asset['sha256']:
                raise ValueError('Example asset checksum mismatch: ' + key)
            (self.input_dir / filename).write_bytes(raw)
        slug = self.spec['slug']
        if slug in {'anthroab', 'abodybuilder3', 'humatch'}:
            heavy, light = self._sequence(p['heavy']), self._sequence(p['light'])
            if slug == 'humatch':
                with (self.input_dir / 'antibodies.csv').open('w', newline='') as handle:
                    writer = csv.writer(handle); writer.writerow(['heavy', 'light']); writer.writerow([heavy, light])
            else:
                name = 'heavy_light.fasta' if slug == 'abodybuilder3' else 'antibodies.fasta'
                (self.input_dir / name).write_text(f'>example_VH\n{heavy}\n>example_VL\n{light}\n')
        elif slug == 'immuscope':
            peptide = self._sequence(p['peptide'])
            if not 9 <= len(peptide) <= 21 or not re.fullmatch(r'[A-Za-z0-9_]+', str(p['allele'])):
                raise ValueError('Use a 9–21 residue peptide and a valid allele identifier.')
            (self.input_dir / 'peptides.txt').write_text(peptide + ' ' + p['allele'] + '\n')
        elif slug == 'ligtmap':
            if '\n' in p['smiles'] or not p['smiles'].strip():
                raise ValueError('Enter one SMILES on a single line.')
            # This teaching example uses the human Kinase class.
            if p['target_class'] != 'Kinase':
                raise ValueError('This example uses target_class=Kinase.')
            (self.input_dir / 'input.smi').write_text(p['smiles'].strip() + '\n')
            (self.input_dir / 'target.lst').write_text('Kinase\n')
        elif slug in {'openfold3', 'protenix'}:
            sequence = self._sequence(p['sequence'])
            if len(sequence) > 150:
                raise ValueError('Use at most 150 residues for this small-runtime example.')
            value = ({'queries': {'ubiquitin_example': {'chains': [{'molecule_type': 'protein', 'chain_ids': ['A'], 'sequence': sequence}]}}}
                     if slug == 'openfold3' else [{'name': 'ubiquitin_example', 'sequences': [{'proteinChain': {'sequence': sequence, 'count': 1}}]}])
            (self.input_dir / ('query.json' if slug == 'openfold3' else 'input.json')).write_text(json.dumps(value, indent=2))
        elif slug == 'rfdiffusion-v3':
            length = p['length']
            if isinstance(length, bool) or not isinstance(length, int) or not 30 <= length <= 100:
                raise ValueError('Use a backbone length from 30 to 100 for this example.')
            (self.input_dir / 'design.json').write_text(json.dumps({'unconditional_example': {'length': f'{length}-{length}'}}))
        for filename in self.spec['required_inputs']:
            path = self.input_dir / filename
            if not path.is_file() or path.stat().st_size == 0:
                raise ValueError('Missing input: ' + filename)
        self.parameters = p
        print('Prepared inputs:', ', '.join(self.spec['required_inputs']))
        return self.input_dir

    def command(self, argv=None):
        return self.udocker + ['run', '--nobanner',
            '--volume=' + str(self.input_dir.resolve()) + ':/work/input',
            '--volume=' + str(self.output_dir.resolve()) + ':/work/output',
            '--env=OMP_NUM_THREADS=2', '--env=MKL_NUM_THREADS=2',
            self.container, *(self.spec['argv'] if argv is None else argv)]

    def run(self, timeout=3600):
        if not self.ready or self.parameters is None:
            raise RuntimeError('Run setup() and prepare_inputs() first.')
        self.completed = False
        log_path = self.run_dir / 'inference.log'
        started = time.monotonic()
        print('Running inference; log:', log_path, flush=True)
        with log_path.open('w') as writer, log_path.open('r') as reader:
            process = subprocess.Popen(self.command(), env=self.env, stdout=writer, stderr=subprocess.STDOUT, start_new_session=True)
            try:
                while process.poll() is None:
                    output = reader.read(8192)
                    if output:
                        print(output, end='', flush=True)
                    if time.monotonic() - started > timeout:
                        raise TimeoutError('Inference exceeded its time limit. See inference.log.')
                    time.sleep(1)
                print(reader.read(), end='', flush=True)
                if process.returncode:
                    raise subprocess.CalledProcessError(process.returncode, self.command())
            finally:
                if process.poll() is None:
                    try:
                        os.killpg(process.pid, signal.SIGTERM)
                    except ProcessLookupError:
                        pass
                    try:
                        process.wait(timeout=10)
                    except subprocess.TimeoutExpired:
                        os.killpg(process.pid, signal.SIGKILL)
                        process.wait()
        self.completed = True
        return self.validate_results()

    @staticmethod
    def validate_file(path):
        name = path.name.lower()
        text = gzip.decompress(path.read_bytes()).decode() if name.endswith('.gz') else path.read_text()
        if name.endswith(('.sdf', '.sdf.gz')):
            if '$$$$' not in text or 'M  END' not in text:
                raise ValueError('Invalid SDF output: ' + str(path))
        elif name.endswith('.pdb'):
            if not any(line.startswith('ATOM  ') for line in text.splitlines()):
                raise ValueError('No atoms in predicted PDB: ' + str(path))
        elif name.endswith('.cif'):
            if '_atom_site.' not in text:
                raise ValueError('No atom_site in predicted mmCIF: ' + str(path))
        elif name.endswith(('.fa', '.fasta')):
            records = text.split('>')[1:]
            if not records or not any(''.join(r.splitlines()[1:]).strip() for r in records):
                raise ValueError('Empty predicted sequences: ' + str(path))
        elif name.endswith('.csv'):
            delimiter = ';' if text.splitlines()[0].count(';') > text.splitlines()[0].count(',') else ','
            rows = list(csv.reader(io.StringIO(text), delimiter=delimiter))
            if len(rows) < 2 or not rows[0] or not any(any(cell.strip() for cell in row) for row in rows[1:]):
                raise ValueError('Empty prediction CSV: ' + str(path))
        return True

    def validate_results(self):
        files = sorted({p for pattern in self.spec['result_globs'] for p in self.output_dir.glob(pattern) if p.is_file() and p.stat().st_size})
        if not files:
            raise RuntimeError('Prediction output was not created. Check inference.log; logs/configuration files alone are not results.')
        for path in files:
            self.validate_file(path)
        if self.spec['slug'] == 'ligtmap':
            status = self.output_dir / 'Output/status.txt'
            if not status.exists() or ':Fail' in status.read_text():
                raise RuntimeError('LigTMap reported a failed calculation; see Output/status.txt.')
        return files

    def archive(self):
        if not self.completed:
            raise RuntimeError('Inference did not complete successfully.')
        files = self.validate_results()
        manifest = {'model': self.spec['slug'], 'image': self.spec['image'] + '@' + self.spec['digest'],
            'gpu': self.gpu_info, 'execution_mode': self.mode, 'host_python': platform.python_version(),
            'parameters': self.parameters, 'argv': self.spec['argv'], 'udocker_version': '1.3.17',
            'prepublication_colab_validation': 'not_performed',
            'inputs': [{'path': name, 'bytes': (self.input_dir / name).stat().st_size, 'sha256': hashlib.sha256((self.input_dir / name).read_bytes()).hexdigest()} for name in self.spec['required_inputs']],
            'reference_assets': {name: {'source': self.assets[key]['source'], 'sha256': self.assets[key]['sha256']} for name, key in self.spec['assets'].items()},
            'outputs': [{'path': str(p.relative_to(self.output_dir)), 'bytes': p.stat().st_size, 'sha256': hashlib.sha256(p.read_bytes()).hexdigest()} for p in files]}
        (self.run_dir / 'run_manifest.json').write_text(json.dumps(manifest, indent=2))
        (self.run_dir / 'image_metadata.json').write_text(self.image_metadata or '{}')
        archive = self.run_dir / (self.spec['slug'] + '_results.zip')
        with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as bundle:
            for path in self.output_dir.rglob('*'):
                if path.is_file():
                    bundle.write(path, str(Path('output') / path.relative_to(self.output_dir)))
            for name in ['run_manifest.json', 'image_metadata.json', 'inference.log']:
                bundle.write(self.run_dir / name, name)
        print('Validated prediction files:', [str(p.relative_to(self.output_dir)) for p in files])
        print('Result archive:', archive)
        return archive


In [ ]:
SPEC = json.loads("{\"slug\":\"lip\",\"image\":\"ghcr.io/connects-scv/laidd-lip\",\"digest\":\"sha256:018c3836fa5827725798ffea5e97416fed306241149ba5411d52cc19514e24fa\",\"compressed_bytes\":6284851109,\"gpu\":true,\"argv\":[\"python\",\"-m\",\"lip\",\"run\",\"--receptor\",\"/work/input/receptor.pdb\",\"--output-dir\",\"/work/output\",\"--device\",\"cuda\",\"--n-steps\",\"5\"],\"parameters\":{},\"assets\":{\"receptor.pdb\":\"complex.pdb\"},\"required_inputs\":[\"receptor.pdb\"],\"result_globs\":[\"**/*.csv\",\"**/*.sdf\"]}")
EXAMPLE_ASSETS = json.loads("{\"complex.pdb\":{\"gzip_base64\":\"...\",\"sha256\":\"1e797e1fd329803cf4abc3427b240683ad33e0387cbfb29a586a616a35bac24c\",\"source\":\"https://files.rcsb.org/download/1IEP.pdb (chain A and STI)\"}}")
EXECUTION_MODE = "F3"  # @param ["F3", "P1", "P2"]
example = ColabModelExample(SPEC, EXAMPLE_ASSETS, execution_mode=EXECUTION_MODE)
print('Model:', SPEC['slug'])
print('Input directory:', example.input_dir)
print('Output directory:', example.output_dir)

## 환경 준비

디스크·GPU를 확인한 뒤 고정 버전의 실행 도구와 모델 이미지를 준비합니다.

In [ ]:
example.setup()

## 입력값 설정

아래 PARAMETERS 값을 수정할 수 있습니다. 구조 파일을 바꾸려면 이 셀을 실행한 후 출력된 input 폴더의 파일을 교체하고, 다음 추론 셀부터 실행하세요.

필수 파일: `receptor.pdb`

- complex.pdb: https://files.rcsb.org/download/1IEP.pdb (chain A and STI) (SHA-256: 1e797e1fd329803cf4abc3427b240683ad33e0387cbfb29a586a616a35bac24c)

In [ ]:
PARAMETERS = json.loads("{}")
example.prepare_inputs(PARAMETERS)

## 실행

실제 예측 명령을 실행합니다. 실패하거나 시간 제한을 넘기면 중단하며, 새 작업 폴더를 사용해 이전 결과가 섞이지 않도록 합니다.

In [ ]:
prediction_files = example.run(timeout=3600)

## 결과 확인·내려받기

구조·서열·CSV 파일의 내용을 검사하고 결과, 실행 로그, 이미지·입력 버전 정보를 ZIP에 담습니다.

In [ ]:
archive = example.archive()
for path in prediction_files:
    print(path.relative_to(example.output_dir), path.stat().st_size, 'bytes')
from IPython.display import display, FileLink
display(FileLink(str(archive)))
DOWNLOAD_RESULTS = True  # @param {type:"boolean"}
if DOWNLOAD_RESULTS:
    try:
        from google.colab import files
    except ImportError:
        print('Local runtime: download the ZIP using the link above.')
    else:
        files.download(str(archive))

## 결과 해석

공결정 리간드가 포함된 사람 ABL 구조로 Lip의 짧은 분자 생성·평가 실습을 실행합니다. 의약품 효능 검증은 아닙니다.

구조·점수는 예측값입니다. 관련 논문과 모델 문서의 지표 정의를 확인하세요. 이 작은 예제의 실행 성공은 논문 전체 실험의 재현이나 플랫폼의 모든 입력에 대한 검증을 뜻하지 않습니다.